# V35: complete recent-turn extraction

The native4,096-token prefix can omit the final turn of long conversations.
The frozen policy retains a suffix of **whole** recent turns in chronological
order if it fits both answer orientations; otherwise the native input stays
unchanged. All retained text still goes through the original formatter. The
proof found312affected original rows out of57,477, with no model quality measured.

The runtime-driven revision freezes a two-pass50/50T1candidate: new-policy Gemma
original and unchanged Llama swapped. There is no reverse/router. This is one
input ablation against the unchanged two-pass baseline0.83053; incumbentH60
0.82987remains a separate comparator. The initial three-pass plan and its
over-budget runtime floor are preserved. No extra hidden pass or prediction cache.

Original votes overlap checkpoint training. A gain here cannot forecast the
official score. A fixed source information gate0.0015/positive paired lower
bound precedes actual full runtime/source qualification and at most one official.
Four training-only examples expose context-loss risks, including code/earlier
setup referred to by a final question. Those examples do not tune this policy.

[Protocol](../docs/SPAN_035_PLAN.md) · [Pre-GPU revision](../docs/SPAN_035_RUNTIME_REVISION.md).
The persistent goal remains actual rank19or better.

In [1]:
from pathlib import Path
import json
import pandas as pd
REPO=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
PUB=REPO/'results/span_035'
for name in ['planning_structure.json','pilot_preparation.json','recent_pilot_cell_rehearsals.json']:
 p=PUB/name
 if p.exists():
  v=json.loads(p.read_text());print(name,{k:x for k,x in v.items() if 'sha256' not in k and k!='rehearsals'})
p=json.loads((PUB/'preparation.json').read_text())
print('Native proof:',{k:v for k,v in p.items() if 'sha256' not in k and k not in ['changed_ids']})
print('No GPU fit/quality follows from CPU staging; those rehearsals use a fake model forward.')
for name in ['push.json','source_verification.json','controller_state.json','progress.json']:
 path=PUB/'jobs/recent_pilot'/name
 if path.exists():
  value=json.loads(path.read_text());print(name,value.get('latest',value))
result=PUB/'H27_results.json'
if not result.exists():print('No audited GPU quality result yet.')
else:
 r=json.loads(result.read_text());print('Audited decision:',r['decision'])
 if r.get('quality_measured'):
  display(pd.DataFrame([dict(model=k,**{n:r[k+'_metrics'][n] for n in ['log_loss','accuracy','macro_f1','brier']}) for k in ['baseline','candidate','prior','hard_majority']]))
  print('Changed-group gain:',r['conditional_changed_group_gain'])
  print('Fixed-prevalence population information:',r['population_information_gain'])
  print('The interval is conditional on affected prompt groups, not unconditional population sampling.')
  classes=[]
  for name in ['baseline','candidate']:
   for i,label in enumerate(['A','B','tie']):classes.append(dict(model=name,label=label,loss=r[name+'_metrics']['class_loss'][i],recall=r[name+'_metrics']['class_recall'][i]))
  display(pd.DataFrame(classes))
  print('Full incumbent-H60 gain established:',r['incumbent_full_gain_established'])
  print('GPU charge/recent inference seconds/memory fractions:',r['charged_gpu_seconds'],r['recent_inference_seconds'],r['peak_memory_fractions'])
 print('Official submission eligible from information pilot:',r['official_submission_eligible'])
for name in ['resource_close.json','final_leaderboard.json']:
 path=PUB/name
 if path.exists():print(name,json.loads(path.read_text()))

planning_structure.json {'at': '2026-11-09T09:21:01.530262+00:00', 'uses_targets': False, 'changed_total': 357, 'single_turn': 42, 'multi_turn': 315, 'turn_count_histogram': {'1': 42, '2': 31, '3': 32, '4': 28, '5': 40, '6': 26, '7': 45, '8': 30, '9': 28, '10': 18, '11': 5, '12': 8, '13': 4, '14': 2, '15': 3, '16': 2, '17': 1, '18': 1, '19': 4, '20': 1, '21': 1, '22': 1, '29': 1, '32': 1, '35': 1, '36': 1}, 'last_turn_char_quantiles': {'0.0': 202.0, '0.25': 2797.0, '0.5': 4012.0, '0.75': 6129.0, '0.9': 14294.4, '1.0': 22140.0}, 'elapsed_seconds': 0.18698190804570913}
pilot_preparation.json {'at': '2026-11-09T09:46:32.299186+00:00', 'prepared': True, 'uses_targets': False, 'rows': 312, 'aggregate_cpu_preparation_seconds': 173.19245958002284, 'stage_seconds': 104.6906036499422, 'new_gpu_executions': 0}
recent_pilot_cell_rehearsals.json {'passed': True, 'actual_serialized_staging_pipeline_final_passed': True, 'standalone_imports_passed': True, 'mode_guards_passed': True, 'notebook_engine_

,model,log_loss,accuracy,macro_f1,brier
0,baseline,0.874040,0.612179,0.588353,0.513001
1,candidate,0.862473,0.647436,0.627065,0.503680
2,prior,1.096107,0.365385,0.178404,0.665012
3,hard_majority,17.535071,0.365385,0.178404,1.269231


Changed-group gain: {'gain': 0.011567364256092245, 'interval_95': [-0.009977906671801737, 0.034360917588608965], 'groups': 312, 'rows': 312, 'replicates': 2000}
Fixed-prevalence population information: {'gain': 6.279064056754494e-05, 'interval_95': [-5.416265430697744e-05, 0.00018651993471555575], 'fixed_changed_fraction': 0.00542825825982567, 'interval_scope': 'Conditional affected-prompt resampling at fixed original population prevalence; not unconditional population sampling'}
The interval is conditional on affected prompt groups, not unconditional population sampling.


,model,label,loss,recall
0,baseline,A,0.798802,0.710526
1,baseline,B,0.756160,0.759615
2,baseline,tie,1.095708,0.329787
3,candidate,A,0.818946,0.719298
4,candidate,B,0.739057,0.807692
5,candidate,tie,1.051806,0.382979


Full incumbent-H60 gain established: False
GPU charge/recent inference seconds/memory fractions: 1338.6699999999983 745.998726226 [0.6991043558010286, 0.5825654661611906]
Official submission eligible from information pilot: False
resource_close.json {'at': '2026-11-09T12:14:04.614785+00:00', 'passed': True, 'accepted_public_versions': 1, 'official_submissions': 0, 'all_owned_jobs_terminal': True, 'public_gpu_charged_seconds': 1338.6699999999983, 'quota_delta_seconds': 1338.6699999999983, 'quota': {'used_seconds': 37989.734, 'reserved_seconds': 0.0, 'allowed_seconds': 108000.0, 'refresh_time': '2026-11-14 00:00:00', 'paid_scaling': False, 'remaining_seconds': 70010.266}, 'inventory_sha256': '8da2970f33953ae09fd2c32063a0de44db48b65df838fa2e6f0d03257aee4b62', 'history_sha256': 'f5e096148e190be53a88a1b399ad2d8364bab85c272288369e23062cc6726e6e', 'leaderboard_sha256': 'a97b93433ba3347cdcf1593aa1d4557d5ef2b2639e62ff743279531807da8e48', 'result_sha256': 'bb665825372674b2352f86c466bd79461ef74b2